# Cleaning Data
**Author:** SUBHADEEP HALDAR
**Track:** Data Analytics — OIBSIP
**Task:** Level 1, Task 3 — Cleaning Data

**Objective:** Take a deliberately messy retail sales dataset and systematically transform
it into a clean, analysis-ready dataset, documenting every decision made along the way.

**Dataset:** Retail Store Sales — Dirty for Data Cleaning (Kaggle), 12,575 rows.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("retail_store_sales.csv")
df.head()


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


## 1. Data Quality Report — Initial Inspection

In [2]:
print("Shape:", df.shape)
print()
print("Data types:")
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
print()
print("Unique values in categorical columns:")
for col in ['Category', 'Payment Method', 'Location', 'Discount Applied']:
    print(f"  {col}: {df[col].unique()}")


Shape: (12575, 11)

Data types:
Transaction ID          str
Customer ID             str
Category                str
Item                    str
Price Per Unit      float64
Quantity            float64
Total Spent         float64
Payment Method          str
Location                str
Transaction Date        str
Discount Applied     object
dtype: object

Missing values per column:
Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

Duplicate rows: 0

Unique values in categorical columns:
  Category: <ArrowStringArray>
[                        'Patisserie',                      'Milk Products',
                           'Butchers',                          'Beverages',
                               'Food',                          'Furniture',
      'Ele

**Data quality summary:** *I found 157 outliers in Total Spent (anything above roughly $377.50), but 
zero outliers in Price Per Unit or Quantity individually. Since both of those underlying 
columns look completely normal, I don't think these are data errors — they're most likely 
genuine large orders where a customer bought a high quantity or picked pricier items, 
pushing the total up naturally. Because of that, I decided to keep these values as they 
are instead of removing or capping them, since they reflect real transactions rather than 
mistakes in the data.*

## 2. Missing Data Handling

This dataset has a useful mathematical relationship: **Total Spent = Quantity × Price Per
Unit**. Wherever two of these three values are present, the third can be calculated exactly
rather than guessed — this is far more accurate than a generic mean/median fill.

In [ ]:
before_nulls = df.isnull().sum()
before_shape = df.shape
before_dtypes = df.dtypes.copy()

mask = df['Total Spent'].isnull() & df['Quantity'].notnull() & df['Price Per Unit'].notnull()
df.loc[mask, 'Total Spent'] = df.loc[mask, 'Quantity'] * df.loc[mask, 'Price Per Unit']
print(f"Recovered {mask.sum()} Total Spent values using Quantity x Price Per Unit")

mask = df['Quantity'].isnull() & df['Total Spent'].notnull() & df['Price Per Unit'].notnull()
df.loc[mask, 'Quantity'] = df.loc[mask, 'Total Spent'] / df.loc[mask, 'Price Per Unit']
print(f"Recovered {mask.sum()} Quantity values using Total Spent / Price Per Unit")

mask = df['Price Per Unit'].isnull() & df['Total Spent'].notnull() & df['Quantity'].notnull()
df.loc[mask, 'Price Per Unit'] = df.loc[mask, 'Total Spent'] / df.loc[mask, 'Quantity']
print(f"Recovered {mask.sum()} Price Per Unit values using Total Spent / Quantity")

print()
print("Remaining nulls after relationship-based recovery:")
print(df[['Price Per Unit', 'Quantity', 'Total Spent']].isnull().sum())


Recovered 0 Total Spent values using Quantity x Price Per Unit
Recovered 0 Quantity values using Total Spent / Price Per Unit
Recovered 609 Price Per Unit values using Total Spent / Quantity

Remaining nulls after relationship-based recovery:
Price Per Unit      0
Quantity          604
Total Spent       604
dtype: int64


**Justification:** Using the Quantity × Price Per Unit relationship recovers exact,
mathematically correct values rather than statistical estimates — this is the most accurate
imputation strategy available here and should always be tried before falling back to
mean/median.

In [ ]:

before_drop = len(df)
df = df.dropna(subset=['Price Per Unit', 'Quantity', 'Total Spent'], how='all')
print(f"Dropped {before_drop - len(df)} rows with all three numeric fields missing")

for col in ['Price Per Unit', 'Quantity', 'Total Spent']:
    remaining = df[col].isnull().sum()
    if remaining > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Filled {remaining} remaining nulls in '{col}' with median ({median_val:.2f})")


Dropped 0 rows with all three numeric fields missing
Filled 604 remaining nulls in 'Quantity' with median (6.00)
Filled 604 remaining nulls in 'Total Spent' with median (108.50)


**Justification:** Rows missing all three numeric fields carry no usable information
to reconstruct a transaction, so dropping them is safer than fabricating three values from
nothing. Any small number of remaining partial nulls are filled with the column median, a
standard, low-bias fallback for numeric data with no better signal available.

In [ ]:
df['Item'] = df['Item'].fillna('Unknown Item')
print(f"Filled {before_nulls['Item']} missing Item values with 'Unknown Item'")

df['Discount Applied'] = df['Discount Applied'].astype('object').fillna('Not Recorded')
print(f"Filled {before_nulls['Discount Applied']} missing Discount Applied values with 'Not Recorded'")


Filled 1213 missing Item values with 'Unknown Item'
Filled 4199 missing Discount Applied values with 'Not Recorded'


**Justification:** `Item` values can't be deduced reliably because several items share
the same price point, so labeling them "Unknown Item" preserves the row without inventing a
false product name. For `Discount Applied`, treating missing values as `False` would bias
discount-rate analysis downward — labeling them "Not Recorded" keeps that distinction honest
and visible to anyone using this data later.

## 3. Duplicate Removal

In [6]:
dupes = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Removed {dupes} duplicate rows")


Removed 0 duplicate rows


**Observation:** No duplicate rows were found in this dataset, so no rows were removed
at this step — but the check is still run and documented for completeness.

## 4. Standardisation — Data Types & Formatting

In [ ]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])

df['Price Per Unit'] = df['Price Per Unit'].astype(float)
df['Quantity'] = df['Quantity'].astype(float)
df['Total Spent'] = df['Total Spent'].astype(float)

df['Transaction ID'] = df['Transaction ID'].astype(str)
df['Customer ID'] = df['Customer ID'].astype(str)

print(df.dtypes)


Transaction ID                 str
Customer ID                    str
Category                       str
Item                           str
Price Per Unit             float64
Quantity                   float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
Discount Applied            object
dtype: object


**Note:** Category, Payment Method, and Location values were already consistently
formatted (checked in Section 1), so no case/spelling standardisation was needed there.
The main fix required was converting `Transaction Date` from plain text to a proper
datetime type, which is essential for any future time-based analysis.

## 5. Outlier Detection

In [8]:
from scipy import stats

for col in ['Price Per Unit', 'Quantity', 'Total Spent']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers)} outliers detected (IQR bounds: {lower:.2f} to {upper:.2f})")


Price Per Unit: 0 outliers detected (IQR bounds: -15.25 to 62.75)
Quantity: 0 outliers detected (IQR bounds: -4.50 to 15.50)
Total Spent: 157 outliers detected (IQR bounds: -138.50 to 377.50)


**Decision:** *Total Spent shows 157 outliers (values above ~$377.50), while Price Per Unit 
and Quantity show none. These high-value transactions are likely legitimate bulk or 
high-quantity purchases rather than data entry errors, since both underlying columns 
(Price Per Unit and Quantity) fall within normal ranges. These outliers are therefore 
retained rather than removed or capped, as they represent real high-value transactions.*

## 6. Before vs. After Summary

In [9]:
after_nulls = df.isnull().sum()
after_shape = df.shape

summary = pd.DataFrame({
    'Column': before_nulls.index,
    'Nulls Before': before_nulls.values,
    'Nulls After': [after_nulls.get(col, 0) for col in before_nulls.index]
})

print("BEFORE vs AFTER CLEANING SUMMARY")
print("="*50)
print(f"Rows before: {before_shape[0]}  |  Rows after: {after_shape[0]}")
print(f"Duplicate rows removed: {dupes}")
print()
print(summary.to_string(index=False))


BEFORE vs AFTER CLEANING SUMMARY
Rows before: 12575  |  Rows after: 12575
Duplicate rows removed: 0

          Column  Nulls Before  Nulls After
  Transaction ID             0            0
     Customer ID             0            0
        Category             0            0
            Item          1213            0
  Price Per Unit           609            0
        Quantity           604            0
     Total Spent           604            0
  Payment Method             0            0
        Location             0            0
Transaction Date             0            0
Discount Applied          4199            0


## 7. Save Cleaned Dataset

In [10]:
df.to_csv("retail_store_sales_cleaned.csv", index=False)
print(f"Cleaned dataset saved: {df.shape[0]} rows, {df.shape[1]} columns, 0 missing values remaining.")
print()
print("Final null check:")
print(df.isnull().sum())


Cleaned dataset saved: 12575 rows, 11 columns, 0 missing values remaining.

Final null check:
Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
dtype: int64


## Conclusion

*- **Total rows cleaned:** 12,575 rows processed, 0 rows dropped, 0 duplicates found.
- **Key decisions made:** Used the mathematical relationship (Total Spent = Quantity × 
  Price Per Unit) to recover 609 missing Price Per Unit values exactly rather than 
  estimating them. Remaining gaps in Quantity and Total Spent (604 each) were filled 
  with the column median as a last resort. Missing Item values (1,213) were labeled 
  "Unknown Item" rather than guessed, and missing Discount Applied values (4,199) were 
  labeled "Not Recorded" instead of assumed False, to avoid biasing discount analysis.
  Transaction Date was converted from text to proper datetime format.
- **Result:** The cleaned dataset has zero missing values and is ready for further 
  analysis, visualization, or modeling.
